# Rhino PoC: metric facial reconstruction

This notebook reconstructs a metric facial surface from a complete Stray Scanner export. It is intentionally linear: run every cell from top to bottom.

Before starting, select **Runtime > Change runtime type > T4 GPU**. The default settings target a complete run in less than one hour, but that target is not yet a measured guarantee.

> Research software only. Outputs are not clinically validated.

## 1. Verify the GPU

The first line must identify an NVIDIA T4. Stop and change the runtime if it does not.

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
print(gpu.stdout.strip())
assert "T4" in gpu.stdout, "This benchmark is configured for a T4 GPU."

## 2. Mount Drive and set paths

Upload the complete Stray Scanner folder to `MyDrive/rhino-poc-data/`. Change only `CAPTURE_NAME` if your folder has another name. Computation stays on `/content`; Drive is used for input and checkpoints.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CAPTURE_NAME = "fbf0b3da11"
DRIVE_INPUT = Path("/content/drive/MyDrive/rhino-poc-data") / CAPTURE_NAME
DRIVE_RESULTS = Path("/content/drive/MyDrive/rhino-poc-results") / CAPTURE_NAME
LOCAL_CAPTURE = Path("/content/input") / CAPTURE_NAME
LOCAL_OUTPUT = Path("/content/work") / CAPTURE_NAME
REPOSITORY = Path("/content/rhino-poc")
MODEL_DIR = Path("/content/models")
FACE_LANDMARKER_MODEL = MODEL_DIR / "face_landmarker.task"

assert DRIVE_INPUT.is_dir(), f"Missing Stray Scanner folder: {DRIVE_INPUT}"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("Input:", DRIVE_INPUT)
print("Results:", DRIVE_RESULTS)

## 3. Install CUDA COLMAP and the project

This cell uses micromamba to install CUDA-enabled COLMAP and an isolated Python 3.11 environment, then clones and installs the project. It does not modify Colab's system Python or preinstalled OpenCV packages. The first run normally takes several minutes.

In [ ]:
import os
import shutil
import subprocess


def run(command):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True)


if not Path("/opt/bin/micromamba").exists():
    run(
        [
            "bash",
            "-lc",
            "cd /opt && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba",
        ]
    )
if not Path("/opt/colmapenv/bin/colmap").exists():
    run(
        [
            "/opt/bin/micromamba",
            "create",
            "-y",
            "-p",
            "/opt/colmapenv",
            "-c",
            "conda-forge",
            "colmap=*=gpu*",
        ]
    )
os.environ["PATH"] = "/opt/colmapenv/bin:" + os.environ["PATH"]

if REPOSITORY.exists():
    run(["git", "-C", REPOSITORY, "fetch", "--quiet", "origin"])
    run(["git", "-C", REPOSITORY, "reset", "--hard", "origin/main"])
else:
    run(["git", "clone", "--quiet", "https://github.com/Daml4Yilmaz/rhino-poc.git", REPOSITORY])

PROJECT_ENV = Path("/opt/rhinoenv")
PROJECT_PYTHON = PROJECT_ENV / "bin" / "python"
if not PROJECT_PYTHON.exists():
    run(
        [
            "/opt/bin/micromamba",
            "create",
            "-y",
            "-p",
            PROJECT_ENV,
            "-c",
            "conda-forge",
            "python=3.11",
            "pip",
        ]
    )
run(
    [
        PROJECT_PYTHON,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-cache-dir",
        str(REPOSITORY),
    ]
)
run([PROJECT_PYTHON, "-m", "pip", "check"])
run(
    [
        PROJECT_PYTHON,
        "-c",
        "import cv2, mediapipe, open3d, pycolmap, trimesh; print('Python dependencies verified')",
    ]
)

help_text = subprocess.run(["colmap", "-h"], capture_output=True, text=True, check=True).stdout
print(help_text.splitlines()[0])
assert "with CUDA" in help_text and "without CUDA" not in help_text, (
    "COLMAP does not have CUDA support."
)
run([PROJECT_PYTHON, "-m", "poc.cli", "download-models", "--output-dir", MODEL_DIR])
assert FACE_LANDMARKER_MODEL.is_file(), "Face landmarker model download failed."
run([PROJECT_PYTHON, "-m", "poc.cli", "--help"])

## 4. Copy and validate the capture

Copying the 80 MB reference capture to local storage avoids thousands of slow Drive reads. Validation uses exported timestamps and frame IDs, not nominal MP4 FPS.

In [ ]:
if LOCAL_CAPTURE.exists():
    shutil.rmtree(LOCAL_CAPTURE)
LOCAL_CAPTURE.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_INPUT, LOCAL_CAPTURE)
run([PROJECT_PYTHON, "-m", "poc.cli", "inspect", LOCAL_CAPTURE, "--json"])

## 5. Build a sparse checkpoint

This stage selects 120 portrait-normalized views, masks the face, and builds the sparse camera solution. Progress is printed every 15 seconds and detailed logs are written under the case output. The completed checkpoint is copied to Drive before dense reconstruction begins.

In [ ]:
if LOCAL_OUTPUT.exists():
    shutil.rmtree(LOCAL_OUTPUT)
run(
    [
        PROJECT_PYTHON,
        "-m",
        "poc.cli",
        "reconstruct",
        LOCAL_CAPTURE,
        "--output",
        LOCAL_OUTPUT,
        "--face-landmarker-model",
        FACE_LANDMARKER_MODEL,
        "--until",
        "sfm",
        "--frame-count",
        "120",
        "--max-dimension",
        "1400",
        "--mvs-references",
        "96",
        "--mvs-source-images",
        "6",
    ]
)
checkpoint = DRIVE_RESULTS / "sparse_checkpoint"
if checkpoint.exists():
    shutil.rmtree(checkpoint)
shutil.copytree(LOCAL_OUTPUT, checkpoint)
print("Sparse checkpoint saved to", checkpoint)

## 6. Run dense reconstruction and measurements

The default run uses photometric PatchMatch with at most 96 reference images and six source images per reference. `--resume` verifies the manifest before skipping the completed sparse stages. A heartbeat remains visible even while COLMAP is silent.

In [ ]:
run(
    [
        PROJECT_PYTHON,
        "-m",
        "poc.cli",
        "reconstruct",
        LOCAL_CAPTURE,
        "--output",
        LOCAL_OUTPUT,
        "--face-landmarker-model",
        FACE_LANDMARKER_MODEL,
        "--resume",
        "--frame-count",
        "120",
        "--max-dimension",
        "1400",
        "--mvs-references",
        "96",
        "--mvs-source-images",
        "6",
        "--no-mvs-geometric",
    ]
)

## 7. Save and inspect results

This cell copies the compact final artifacts and logs to Drive, prints measurements and stage timings, and offers the GLB for download. Preserve `case.json`, `run.log`, and `colmap.log` with every benchmark.

In [ ]:
import json

final_dir = DRIVE_RESULTS / "final"
final_dir.mkdir(parents=True, exist_ok=True)
artifacts = [
    "case.json",
    "run.log",
    "face_mesh_raw.ply",
    "scale.json",
    "face_model.glb",
    "landmarks.json",
    "measurements.json",
]
for name in artifacts:
    source = LOCAL_OUTPUT / name
    if source.exists():
        shutil.copy2(source, final_dir / name)
colmap_log = LOCAL_OUTPUT / "colmap" / "colmap.log"
if colmap_log.exists():
    shutil.copy2(colmap_log, final_dir / "colmap.log")

for name in ("scale.json", "measurements.json"):
    path = LOCAL_OUTPUT / name
    if path.exists():
        print(f"\n{name}:")
        print(json.dumps(json.loads(path.read_text()), indent=2))
manifest = json.loads((LOCAL_OUTPUT / "case.json").read_text())
print("\nStage times:")
for stage, record in manifest["stages"].items():
    print(
        f"  {stage:10s} {record.get('metadata', {}).get('elapsed_seconds', 0):8.1f}s  {record['status']}"
    )
print("\nSaved to", final_dir)

from google.colab import files

if (LOCAL_OUTPUT / "face_model.glb").exists():
    files.download(str(LOCAL_OUTPUT / "face_model.glb"))